# 63장 · 같은 프로젝트를 에이전틱 서비스로 묶기

‘과제는 어디에 내나요?’라는 질문을 받았습니다. 이제까지 배운 검색·근거 표시·사용자별 대화를 한 서비스 함수에서 이어 봅니다. 질문과 출처는 기존 과정 안내 예제를 그대로 사용합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.

## 검색 함수에 원문 한 건 전달하기

문서 번호와 페이지는 역할이 다릅니다. course-guide-v1:p2는 자료를 구별하는 ID이고 이 문장이 인쇄된 쪽은 1쪽입니다. 답변과 출처를 같은 원문에서 가져오면 다음 단계에서 다시 대조할 수 있습니다.



**볼 곳**　‘검색 질문’에서 출발해 ‘과제함 문단’를 거친 뒤 ‘원문 전달’가 되는 순서를 따라가세요.

**애니메이션의 의미**　질문과 함께 실제 검색 함수가 읽을 원문을 전달합니다.


### 셀 01 · 가상 과정 운영 안내에서 찾기

한 문서만 있는 작은 검색입니다. 저장한 자료를 어떤 질문에 반환할지 확인하세요.

In [1]:
document = {"id": "course-guide-v1:p2", "source": "가상 과정 운영 안내 · 45일 수업용 페이지판 (source-pages.html)",
            "page": 1, "text": "과제는 학습 관리 페이지의 과제함에 제출합니다. 제출 뒤 파일 이름과 등록 시간을 확인합니다."}

def search(question):
    return [document] if "과제" in question else []

print(search("과제는 어디에 내나요?")[0]["text"])

과제는 학습 관리 페이지의 과제함에 제출합니다. 제출 뒤 파일 이름과 등록 시간을 확인합니다.


과제함이라는 답과 제출 후 확인할 두 항목을 모두 원문에서 찾을 수 있습니다. 여기서 쓰는 검색은 키워드 하나만 보는 수업용 함수입니다.

## 서비스가 처리한 결과와 답변을 나누기

정상 실행에도 근거가 없을 수 있습니다. 앱이 읽는 run_status에는 실행 상황을, result.status에는 답변을 만들었는지 적습니다. 소유 대화인지 확인한 뒤에만 검색하고, 빈 결과에서는 생성 횟수를 늘리지 않습니다.



**볼 곳**　‘서비스 결과’에서 갈라지는 두 경우를 비교하세요. 위쪽은 ‘처리 상태’, 아래쪽은 ‘표시할 답’입니다.

**애니메이션의 의미**　처리 상태와 사용자에게 보여 줄 답을 따로 읽습니다.


### 셀 02 · 한 질문을 끝까지 처리하기

제공 서비스의 읽기 경로를 줄인 함수입니다. 이 요청의 답과 출처 쪽수를 확인하세요.

소유자가 맞지 않으면 조회를 계속하기 전에 거부합니다. 허용된 소유자에게서 검색이 빈 경우에는 근거 부족 결과로 끝납니다. 문서를 찾은 경우에만 그 원문을 건네 답을 만듭니다. 세 경우의 마지막 줄을 각각 찾아보세요.

In [2]:
owners = {"chat-a": "alice"}
def answer(question, *, owner, conversation):
    if owners.get(conversation) != owner:
        raise PermissionError("conversation_not_owned")
    docs = search(question)
    if not docs:
        return {"run_status": "completed", "result": {"status": "no_evidence", "answer": "자료에서 확인할 수 없습니다.", "sources": [], "generation_calls": 0}}
    doc = docs[0]
    result = {"status": "answered", "answer": f"{doc['text']} [{doc['id']}]",
              "sources": [doc], "generation_calls": 1}
    return {"run_status": "completed", "result": result}

row = answer("과제는 어디에 내나요?", owner="alice", conversation="chat-a")
print(row["result"]["answer"])
print(row["result"]["sources"][0]["page"])

과제는 학습 관리 페이지의 과제함에 제출합니다. 제출 뒤 파일 이름과 등록 시간을 확인합니다. [course-guide-v1:p2]
1


답변 끝에 문서 ID가 붙고 다음 줄에는 1이 나옵니다. generation_calls는 이 축약 예제의 고정 답변 생성 경로에 들어간 횟수입니다. 실제 모델 호출은 없습니다.

### 과제 03 · 안내문에 없는 주차 질문 처리

같은 사용자가 주차비를 물었습니다. 결과 상태와 생성 횟수를 꺼내세요.

In [3]:
# answer에 주차 질문을 전달하고 result를 확인하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>missing = answer(&#x27;주차는 무료인가요?&#x27;, owner=&#x27;alice&#x27;, conversation=&#x27;chat-a&#x27;)[&#x27;result&#x27;]
print(missing[&#x27;status&#x27;], missing[&#x27;generation_calls&#x27;])</code></pre>
<p>실행 결과</p><pre><code>no_evidence 0</code></pre>
<p>no_evidence 0입니다. 주차비가 0원이라는 답과는 다릅니다. DB나 MCP 없이도 이 질문에서 필요한 기능을 완성할 수 있습니다.</p>
</details>

## 돌아보기

이 서비스에 문의 등록 버튼을 붙이면 데이터를 바꾸는 일이 생깁니다. 누가 무슨 내용을 등록하려는지 보여 주는 단계는 어디에 넣을지 생각해 보세요.

### 필수 검색의 결과와 종료



**볼 곳**　다른 소유자, 빈 검색 결과, 근거가 있는 검색 결과가 각각 어디서 끝나는지 보세요.

**애니메이션의 의미**　필수 검색을 쓰더라도 권한 검사와 근거 유무에 따라 서로 다른 위치에서 종료됩니다.
